Aim: To market a new cleaning product to customers

In [297]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [298]:
#Link to dataset: https://www.kaggle.com/datasets/jillwang87/online-retail-ii
#loading dataframes
df1 = pd.read_csv('online_retail_09_10.csv')
df2 = pd.read_csv('online_retail_10_11.csv')

In [299]:
#merging dataframes
df = pd.concat([df1,df2])

In [300]:
#sort values by invoice date
df.sort_values('InvoiceDate', inplace=True)

Data Cleaning

In [301]:
#first removing rows with cancelled orders
df = df[~df['InvoiceNo'].str.contains('C')]

In [302]:
#checking null values
df.isnull().sum()


,0
InvoiceNo,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,242257
Country,0


In [303]:
#removing only those rows that contain no customer ID and description
df = df.drop(df[df['Description'].isnull() & df['CustomerID'].isnull()].index)


In [304]:
#Checking for negative values in the quantity column
df.sort_values('Quantity')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
225530,556691,23005,printing smudges/thrown away,-9600,6/14/2011 10:37,0.00,NaN,United Kingdom
225529,556690,23005,printing smudges/thrown away,-9600,6/14/2011 10:37,0.00,NaN,United Kingdom
225528,556687,23003,Printing smudges/thrown away,-9058,6/14/2011 10:36,0.00,NaN,United Kingdom
428975,530348,16235,?,-9000,11/2/2010 15:48,0.00,NaN,United Kingdom
194372,507913,10120,Zebra invcing error,-9000,5/11/2010 17:16,0.00,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,3/17/2010 13:09,0.10,13902.0,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,3/17/2010 13:09,0.10,13902.0,Denmark
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2/15/2010 11:57,0.10,13902.0,Denmark
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1/18/2011 10:01,1.04,12346.0,United Kingdom


In [305]:
#removal of rows with negative and 0 values in the quantity column

df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

In [306]:
#standardizing time

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

import datetime as dt
df['InvoiceDate'] = df['InvoiceDate'].dt.strftime('%d/%m/%Y')

In [307]:
#Creating new column, total revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [308]:
df[df['Revenue'] >1]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
48948,493971,20751,FUNKY WASHING UP GLOVES ASSORTED,1,10/01/2010,2.10,13259.0,United Kingdom,2.10
48923,493971,22122,SET OF 2 FANCY FONT TEA TOWELS,2,10/01/2010,2.95,13259.0,United Kingdom,5.90
48922,493971,22124,SET OF 2 TEA TOWELS PING MICROWAVE,2,10/01/2010,2.95,13259.0,United Kingdom,5.90
48921,493971,84327A,PINK JUMPER LARRY THE LAMB,1,10/01/2010,2.10,13259.0,United Kingdom,2.10
48920,493971,84580,MOUSE TOY WITH PINK T-SHIRT,1,10/01/2010,3.75,13259.0,United Kingdom,3.75
...,...,...,...,...,...,...,...,...,...
332549,566079,20838,FRENCH LATTICE CUSHION COVER,12,09/09/2011,0.85,17593.0,United Kingdom,10.20
332550,566079,22400,MAGNETS PACK OF 4 HOME SWEET HOME,24,09/09/2011,0.39,17593.0,United Kingdom,9.36
332551,566079,22396,MAGNETS PACK OF 4 RETRO PHOTO,24,09/09/2011,0.39,17593.0,United Kingdom,9.36
332572,566079,22924,FRIDGE MAGNETS LA VIE EN ROSE,24,09/09/2011,0.85,17593.0,United Kingdom,20.40


In [309]:
df.sort_values(by = 'Revenue', inplace = True)

Feature Engineering

In [310]:
#Total spend per customer
df_totalSpend = pd.pivot_table(df, index = 'CustomerID', values = 'Revenue', aggfunc = 'sum')
df_totalSpend.reset_index(inplace = True)
df_totalSpend.rename(columns = {'Revenue':'TotalSpend'}, inplace = True)
df_totalSpend.sort_values(by = 'TotalSpend', ascending = False)
df_totalSpend['TotalSpend'] = df_totalSpend['TotalSpend'].round(2)

In [311]:
#Average value of orders per customer
df_avgOrderValue = pd.pivot_table(df, index = ['CustomerID'], values = 'Revenue', aggfunc = 'mean')
df_avgOrderValue.reset_index(inplace = True)
df_avgOrderValue.rename(columns = {'Revenue':'Average Customer Spend'}, inplace = True)
df_avgOrderValue.sort_values(by = 'Average Customer Spend', ascending = False)
df_avgOrderValue['Average Customer Spend'] = df_avgOrderValue['Average Customer Spend'].round(2)


In [312]:
#Number of invoices per customer
df_invoice = pd.pivot_table(df, index = ['CustomerID'], values = 'InvoiceNo', aggfunc = 'nunique')
df_invoice.reset_index(inplace = True)
df_invoice.rename(columns = {'InvoiceNo':'Number of Invoices'}, inplace = True)
df_invoice.sort_values(by = 'Number of Invoices', ascending = False)

,CustomerID,Number of Invoices
2538,14911.0,398
400,12748.0,336
5432,17841.0,211
2935,15311.0,208
739,13089.0,203
...,...,...
3308,15686.0,1
3310,15688.0,1
49,12396.0,1
51,12398.0,1


In [313]:
#combining all these tables together
df_combined = pd.merge(df_totalSpend, df_avgOrderValue, on = 'CustomerID')
df_combined = pd.merge(df_combined, df_invoice, on = 'CustomerID')

In [314]:
df_combined.describe()

,CustomerID,TotalSpend,Average Customer Spend,Number of Invoices
count,5877.000000,5877.000000,5877.000000,5877.000000
mean,15315.168964,3007.041060,67.395171,6.285350
std,1715.682828,14679.214834,2213.598449,13.005016
min,12346.000000,2.950000,2.020000,1.000000
25%,13833.000000,345.000000,11.470000,1.000000
50%,15314.000000,893.590000,17.350000,3.000000
75%,16798.000000,2296.720000,24.190000,7.000000
max,18287.000000,603919.730000,168469.600000,398.000000


In [315]:
#calculation of recency - by calculating the number of days that have elapsed between last date of purchase
#and second last date of purchase.
df_recency = pd.pivot_table(df, index = 'CustomerID', values = 'InvoiceDate', aggfunc = 'max')
df_recency.reset_index(inplace = True)
df_recency.rename(columns = {'InvoiceDate':'LastPurchaseDate'}, inplace = True)
df_recency['LastPurchaseDate'] = pd.to_datetime(df_recency['LastPurchaseDate'])
df_recency['LastPurchaseDate'] = df_recency['LastPurchaseDate'].dt.strftime('%d/%m/%Y')

/tmp/ipython-input-4278219020.py:6: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_recency['LastPurchaseDate'] = pd.to_datetime(df_recency['LastPurchaseDate'])


In [316]:
#calculation of recency
today = dt.datetime(2012,1,1)
df_recency['DaysSinceLastPurchase'] = (today - pd.to_datetime(df_recency['LastPurchaseDate'])).dt.days

/tmp/ipython-input-1800253919.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_recency['DaysSinceLastPurchase'] = (today - pd.to_datetime(df_recency['LastPurchaseDate'])).dt.days


In [317]:
df_recency.describe()

,CustomerID,DaysSinceLastPurchase
count,5877.000000,5877.000000
mean,15315.168964,352.849073
std,1715.682828,219.891439
min,12346.000000,23.000000
25%,13833.000000,128.000000
50%,15314.000000,398.000000
75%,16798.000000,530.000000
max,18287.000000,761.000000


In [318]:
df_combined = pd.merge(df_combined, df_recency, on = 'CustomerID')

In [319]:
df_combined.describe()

,CustomerID,TotalSpend,Average Customer Spend,Number of Invoices,DaysSinceLastPurchase
count,5877.000000,5877.000000,5877.000000,5877.000000,5877.000000
mean,15315.168964,3007.041060,67.395171,6.285350,352.849073
std,1715.682828,14679.214834,2213.598449,13.005016,219.891439
min,12346.000000,2.950000,2.020000,1.000000,23.000000
25%,13833.000000,345.000000,11.470000,1.000000,128.000000
50%,15314.000000,893.590000,17.350000,3.000000,398.000000
75%,16798.000000,2296.720000,24.190000,7.000000,530.000000
max,18287.000000,603919.730000,168469.600000,398.000000,761.000000


In [320]:
#recency score designation

df_combined['Recency Score'] = None

for idx, row in df_combined.iterrows():
    if row.DaysSinceLastPurchase <= 100:
        df_combined.at[idx, 'Recency Score'] = 1
    elif row.DaysSinceLastPurchase <= 200 and row.DaysSinceLastPurchase >= 100:
        df_combined.at[idx, 'Recency Score'] = 2
    elif row.DaysSinceLastPurchase <= 300 and row.DaysSinceLastPurchase >= 200:
        df_combined.at[idx, 'Recency Score'] = 3
    elif row.DaysSinceLastPurchase <= 400 and row.DaysSinceLastPurchase >=300:
        df_combined.at[idx, 'Recency Score'] = 4
    elif row.DaysSinceLastPurchase <= 500 and row.DaysSinceLastPurchase >= 400:
        df_combined.at[idx, 'Recency Score'] = 5
    elif row.DaysSinceLastPurchase <= 600 and row.DaysSinceLastPurchase >= 500:
        df_combined.at[idx, 'Recency Score'] = 6
    elif row.DaysSinceLastPurchase <= 700 and row.DaysSinceLastPurchase >= 600:
        df_combined.at[idx, 'Recency Score'] = 7
    else:
        df_combined.at[idx, 'Recency Score'] = 8

In [321]:
df_combined['Recency Score Adjective'] = None

In [322]:
#Creating a column to denote the recency score.
df_combined['Recency Score Adjective'] = df_combined['Recency Score'].map(
    {1 : 'Less than 100 Days',
     2: 'Between 100 and 200 Days',
     3: 'Between 200 and 300 Days',
     4: 'Between 300 and 400 Days',
     5: 'Between 400 and 500 Days',
     6: 'Between 500 and 600 Days',
     7: 'Between 600 and 700 Days',
     8: 'Greater than 700 Days'}
)

In [323]:
df_combined['Recency Score Adjective'].value_counts()

,count
Recency Score Adjective,
Less than 100 Days,1297
Between 400 and 500 Days,1207
Between 600 and 700 Days,739
Between 100 and 200 Days,643
Between 500 and 600 Days,629
Between 200 and 300 Days,545
Between 300 and 400 Days,533
Greater than 700 Days,284


In [324]:
import re

cleaning_keywords = [
    "clean", "cleaner", "cleaning", "detergent", "disinfect",
    "soap", "washing", "laundry", "bleach", "sanitiser", "sanitizer",
    "spray", "wipe", "wipes", "polish", "scrub", "descaler",
    "dish", "dishwashing", "floor", "surface", "toilet", "bathroom",
    "kitchen"
]

household_keywords = [
    "house", "home", "household", "kitchen", "bathroom", "bedroom",
    "storage", "container", "organiser", "organizer", "bin",
    "basket", "rack", "holder", "shelf", "tray", "box",
    "towel", "cloth", "sponge", "mat", "liner"
]

reusable_keywords = [
    "reusable", "eco", "eco-friendly", "sustainable", "green",
    "biodegradable", "compostable", "recycled", "recyclable",
    "zero waste", "plastic free", "refill", "refillable",
    "glass", "bamboo", "cotton", "linen"
]

keywords = cleaning_keywords + household_keywords + reusable_keywords

df['Description'] = df['Description'].str.lower()

eco_keywords = '|'.join(map(re.escape, keywords))#joins all keywords and escapes special characters with '|' between each word.

In [325]:
df['Eco-Interest Flag'] = None
df['Eco-Interest Flag'] = df['Description'].str.contains(eco_keywords)

In [326]:
#isolating to those customer who have an interest flag
df_ecoInterest = df[df['Eco-Interest Flag'] == True]

In [327]:
#creating pivot table to summarise customer data
df_ecoInterestFlag = pd.pivot_table(df_ecoInterest, index = 'CustomerID',values = 'Eco-Interest Flag', aggfunc = 'sum')
df_ecoInterestFlag.reset_index(inplace = True)
df_ecoInterestFlag.rename(columns = {'Eco-Interest Flag':'Eco-Interest Flag Count'}, inplace = True)

In [328]:
df_combined = pd.merge(df_combined, df_ecoInterestFlag, on = 'CustomerID', how = 'outer')

In [329]:
pd.set_option('display.max_rows', 10)
df_combined['Eco-Interest Flag Count'].value_counts()

,count
Eco-Interest Flag Count,
1.0,282
3.0,280
2.0,260
4.0,250
6.0,216
...,...
223.0,1
440.0,1
243.0,1


In [330]:
df_combined.loc[df_combined['Eco-Interest Flag Count'].isna(), 'Eco-Interest Flag Count'] = 0

In [331]:
df_combined

,CustomerID,TotalSpend,Average Customer Spend,Number of Invoices,LastPurchaseDate,DaysSinceLastPurchase,Recency Score,Recency Score Adjective,Eco-Interest Flag Count
0,12346.0,77556.46,2281.07,12,28/06/2010,552,6,Between 500 and 600 Days,20.0
1,12347.0,5581.32,22.33,8,31/10/2011,62,1,Less than 100 Days,42.0
2,12348.0,1791.96,39.82,4,27/09/2010,461,5,Between 400 and 500 Days,0.0
3,12349.0,4228.69,24.30,3,29/04/2010,612,7,Between 600 and 700 Days,61.0
4,12350.0,239.20,18.40,1,02/02/2011,333,4,Between 300 and 400 Days,4.0
...,...,...,...,...,...,...,...,...,...
5872,18283.0,2730.69,2.78,22,30/11/2011,32,1,Less than 100 Days,129.0
5873,18284.0,461.68,16.49,1,04/10/2010,454,5,Between 400 and 500 Days,7.0
5874,18285.0,427.00,35.58,1,17/02/2010,683,7,Between 600 and 700 Days,8.0
5875,18286.0,1296.43,19.35,2,20/08/2010,499,5,Between 400 and 500 Days,17.0


In [333]:
df.to_csv('full dataframe.csv')
df_combined.to_csv('key insights.csv')